# Step 4 — Fused FlashAttention-1 (v4_fused) — run-of-record (vast.ai T4)

Single fused pass: warp-per-row, staged K/V in smem, register-resident O, online softmax with
the **O-rescale**. Keeps v3's S-off-HBM property AND restores v2's GEMM-like parallelism.

**Thesis to test:** v4 should *beat v2* in wall-clock (reversing v3's 3-7x regression) while
keeping the +17 MB peak-memory footprint (S still gone). Counter-free throughout (ncu blocked
on cloud rentals): peak-memory for the S proof, torch-profiler CUPTI for per-kernel timing.


## 0. Environment (cu124 torch + python symlink already done on this image)


In [ ]:
!nvcc --version


In [ ]:
!pip install ninja pytest -q


In [ ]:
!python -c "import torch; print('cuda_ok', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', torch.version.cuda)"


## 1. Pull the v4 code


In [ ]:
!git pull origin main


## 2. Predict the roofline BEFORE running (record the floor — the deliverable is the distance)


In [ ]:
!python -m roofline.predict --arch sm_75 --shape 1x8x8192x64 --precision fp32
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x128 --precision fp32


## 3. Build smoke (JIT compile v4 + one forward). A clean compile here = the kernel built.


In [ ]:
# Clear any stale JIT cache from a prior build, then compile v4 via one forward.
import shutil, os, glob
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v4_fused')):
    shutil.rmtree(d, ignore_errors=True)
import torch
from fa_kernels import attention
q=torch.randn(1,8,512,64,device='cuda'); k=torch.randn_like(q); v=torch.randn_like(q)
out=attention(q,k,v,backend='v4_fused'); torch.cuda.synchronize()
print('v4 built + ran, out shape', tuple(out.shape))


## 4. Correctness vs SDPA (atol/rtol 1e-4) — full sweep + the long-N O-rescale stability test


In [ ]:
!python -m pytest tests/test_correctness.py -k v4_fused -v


## 5. S-elimination proof (peak memory) — v4 must match v3's +17 MB, NOT v2's +2164 MB
The O-rescale lets v4 stay single-pass with no S and no per-row (m,l) HBM scratch — even less
than v3's two tiny stats arrays.


In [ ]:
%%writefile mem_check.py
import torch
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
for backend in ["v2_tiled", "v3_online", "v4_fused"]:
    q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); base=torch.cuda.memory_allocated()
    out=attention(q,k,v,backend=backend); torch.cuda.synchronize()
    print(f"{backend}: peak +{(torch.cuda.max_memory_allocated()-base)/1e6:.1f} MB  (a materialized S = {B*H*N*N*4/1e6:.0f} MB)")
    del q,k,v,out; torch.cuda.empty_cache()


In [ ]:
!python mem_check.py


## 6. CUPTI per-kernel trace — v4 is a SINGLE fused kernel; measure its distance from the 17 ms floor
v3 was pass2-dominated (88.6%) and 151x above the floor. v4 collapses to one kernel — read its
total CUDA time and compare to the 16.97 ms MMA lower bound.


In [ ]:
%%writefile prof_check.py
import torch
from torch.profiler import profile, ProfilerActivity
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
for _ in range(3): attention(q,k,v,backend="v4_fused")   # warmup + JIT
torch.cuda.synchronize()
with profile(activities=[ProfilerActivity.CUDA]) as prof:
    for _ in range(10): attention(q,k,v,backend="v4_fused")
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))


In [ ]:
!python prof_check.py


## 7. Bench vs SDPA — the headline. Run v4, then v3 and v2 for the apples-to-apples comparison.
Win condition: v4/SDPA speedup >= v2/SDPA at matching shapes (v4 beats v2), and v4 >> v3.


In [ ]:
!python -m bench.harness --backend v4_fused --precision fp32


In [ ]:
!python -m bench.harness --backend v3_online --precision fp32


In [ ]:
!python -m bench.harness --backend v2_tiled --precision fp32


## 8. (Optional) causal sweep — exercises the early-out mask path in the fused loop


In [ ]:
!python -m bench.harness --backend v4_fused --precision fp32 --causal
